# 训练演示 Notebook

本 Notebook 演示 Transformer 模型的训练过程，包括：
1. 加载数据和构建模型
2. 训练几个 epoch 并观察 loss 变化
3. 绘制训练曲线
4. 观察生成文本质量的变化

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))
os.chdir(os.path.dirname(os.getcwd()))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import json
import math

from src.transformer import Transformer
from src.dataset import load_data, PAD_ID
from src.train import train_one_epoch, evaluate, generate_sample

print('导入完成！')

## 1. 加载数据

In [ ]:
# 加载数据集
train_dataset, val_dataset, tokenizer = load_data('data', seq_len=64)

# 创建 DataLoader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# 查看一个样本
src, tgt = train_dataset[0]
print(f'源序列: {tokenizer.decode(src.tolist())}')
print(f'目标序列: {tokenizer.decode(tgt.tolist())}')

## 2. 构建模型

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')

model = Transformer(
    vocab_size=tokenizer.vocab_size,
    d_model=128,
    num_heads=4,
    d_ff=512,
    num_layers=2,
    dropout=0.1,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'模型参数量: {total_params:,}')

## 3. 训练模型

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 种子文本
with open('data/input.txt', 'r') as f:
    seed_text = f.read()[:64]

# 训练循环
epochs = 10
train_losses = []
val_losses = []
samples = []

for epoch in range(1, epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, epoch, log_interval=200)
    val_loss = evaluate(model, val_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    # 每 2 个 epoch 生成一个样本
    if epoch % 2 == 0:
        sample = generate_sample(model, tokenizer, device, seed_text, max_len=100)
        samples.append((epoch, sample))
    
    print(f'Epoch {epoch}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')

## 4. 绘制训练曲线

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss 曲线
ax1.plot(range(1, epochs + 1), train_losses, 'b-o', label='Train Loss', markersize=4)
ax1.plot(range(1, epochs + 1), val_losses, 'r-o', label='Val Loss', markersize=4)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training & Validation Loss', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# PPL 曲线
train_ppls = [math.exp(min(l, 20)) for l in train_losses]
val_ppls = [math.exp(min(l, 20)) for l in val_losses]
ax2.plot(range(1, epochs + 1), train_ppls, 'b-o', label='Train PPL', markersize=4)
ax2.plot(range(1, epochs + 1), val_ppls, 'r-o', label='Val PPL', markersize=4)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Perplexity', fontsize=12)
ax2.set_title('Training & Validation Perplexity', fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('图片已保存到 notebooks/training_curves.png')

## 5. 观察生成文本质量变化

In [ ]:
print('生成文本随训练进度的变化：\n')
for epoch, sample in samples:
    print(f'--- Epoch {epoch} ---')
    print(sample[:150])
    print()

## 6. 也可以加载已有的训练日志绘制曲线

In [ ]:
# 如果有已训练好的日志
log_path = 'experiments/baseline/train_log.json'
if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        log = json.load(f)
    
    epochs_log = [e['epoch'] for e in log]
    train_l = [e['train_loss'] for e in log]
    val_l = [e['val_loss'] for e in log]
    
    plt.figure(figsize=(10, 5))
    plt.plot(epochs_log, train_l, 'b-o', label='Train Loss', markersize=3)
    plt.plot(epochs_log, val_l, 'r-o', label='Val Loss', markersize=3)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Full Training History (from saved log)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print(f'未找到训练日志: {log_path}')
    print('请先运行 python -m src.train 完成训练。')